# Agent Action Distribution Plots

This notebook focuses on action usage rather than survival. It reads the local W&B full-history cache and plots:

- fraction of action 0 / do-nothing actions by agent,
- non-idle action fraction by agent,
- joint distribution of how many agents act at the same environment step,
- illegal action rates by agent,
- optional exact action-id histograms when rollout action-trace tables exist.


In [ ]:
from pathlib import Path
import json
import os
import re
import warnings
from typing import Iterable, Sequence

import numpy as np
import pandas as pd

try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
except ImportError as exc:
    raise ImportError("Install plotly first, for example: pip install plotly") from exc

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 160)

cwd = Path.cwd().resolve()
if (cwd / "main.py").is_file():
    TASK_DIR = cwd
elif (cwd / "Topology_Task" / "main.py").is_file():
    TASK_DIR = cwd / "Topology_Task"
elif (cwd.parent / "main.py").is_file():
    TASK_DIR = cwd.parent
else:
    raise RuntimeError("Could not locate Topology_Task/main.py from the current working directory.")

CACHE_DIR = TASK_DIR / "outputs" / "wandb_cache"
FULL_HISTORY_DIR = CACHE_DIR / "full_history"
CACHE_INDEX_PATH = CACHE_DIR / "full_history_cache_index.csv"
FIG_DIR = TASK_DIR / "outputs" / "action_distribution_figures"
TRACE_CACHE_DIR = CACHE_DIR / "action_trace_tables"
for directory in [FIG_DIR, TRACE_CACHE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Task dir: {TASK_DIR}")
print(f"Cache index: {CACHE_INDEX_PATH}")
print(f"Figure dir: {FIG_DIR}")


## Configuration

The default regex targets the GINE `s0/s1/s2` families, including the `_neighbors_` reruns. Set `RUN_NAME_REGEX = None` to load every cached run.

In [ ]:
# Local cached histories to include.
RUN_NAME_REGEX = r"^best_(?:000|00|01|02|03|04|05|06|07|08)_(?:neighbors_)?(?:shared|nonshared)_actor_gnn"
EXCLUDE_RUN_NAME_REGEX = None
MAX_RUNS = None

# Plot behavior.
SMOOTH_WINDOW = 5
FINAL_WINDOW_STEPS = 10
PLOT_STEP_MAX_M = None  # e.g. 30 to stop plots at 30M steps.
SAVE_FIGURES = True
SHOW_FIGURES = True

# Optional exact action-id traces. This requires runs launched with trace_rollout_actions=true.
FETCH_TRACE_TABLES_FROM_WANDB = False
ENTITY = os.getenv("WANDB_ENTITY", "corentin-plumet-epfl")
PROJECT = os.getenv("WANDB_PROJECT", "Grid2Op")
WANDB_API_TIMEOUT = 300
TRACE_TABLE_KEYS = (
    "train/rollout_action_trace",
    "train/rollout_action_trace_table",
    "test/rollout_action_trace",
    "test/rollout_action_trace_table",
    "eval/rollout_action_trace",
    "eval/rollout_action_trace_table",
)


## Load Cached Histories

In [ ]:
def safe_name(text):
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", str(text)).strip("._-")
    return text or "plot"


def save_figure(fig, name):
    if not SAVE_FIGURES or fig is None:
        return None
    path = FIG_DIR / f"{safe_name(name)}.html"
    fig.write_html(path, include_plotlyjs="cdn")
    print(f"Saved: {path}")
    return path


def _compile_regex(pattern):
    return re.compile(pattern) if pattern else None


def load_cache_index(path=CACHE_INDEX_PATH):
    if not Path(path).exists():
        raise FileNotFoundError(
            f"Missing {path}. Run the W&B history download/cache notebook first."
        )
    index = pd.read_csv(path)
    required = {"name", "id"}
    missing = sorted(required - set(index.columns))
    if missing:
        raise ValueError(f"Cache index is missing required columns: {missing}")
    index = index.rename(columns={"name": "run_name", "id": "run_id"}).copy()
    for col in ["history_parquet", "history_csv"]:
        if col not in index.columns:
            index[col] = None
    index["history_parquet"] = index["history_parquet"].apply(lambda value: Path(value) if pd.notna(value) else None)
    index["history_csv"] = index["history_csv"].apply(lambda value: Path(value) if pd.notna(value) else None)
    index["has_history"] = index.apply(
        lambda row: bool(row["history_parquet"] and row["history_parquet"].exists())
        or bool(row["history_csv"] and row["history_csv"].exists()),
        axis=1,
    )
    return index[index["has_history"]].reset_index(drop=True)


def select_cached_runs(index, include_pattern=RUN_NAME_REGEX, exclude_pattern=EXCLUDE_RUN_NAME_REGEX, max_runs=MAX_RUNS):
    selected = index.copy()
    include_re = _compile_regex(include_pattern)
    exclude_re = _compile_regex(exclude_pattern)
    if include_re is not None:
        selected = selected[selected["run_name"].astype(str).str.contains(include_re, na=False)]
    if exclude_re is not None:
        selected = selected[~selected["run_name"].astype(str).str.contains(exclude_re, na=False)]
    selected = selected.sort_values(["run_name", "run_id"]).reset_index(drop=True)
    if max_runs is not None:
        selected = selected.head(int(max_runs))
    if selected.empty:
        raise RuntimeError("No cached runs matched RUN_NAME_REGEX / EXCLUDE_RUN_NAME_REGEX.")
    return selected


def read_cached_history(row):
    parquet_path = row.get("history_parquet")
    csv_path = row.get("history_csv")
    if parquet_path and Path(parquet_path).exists():
        history = pd.read_parquet(parquet_path)
    elif csv_path and Path(csv_path).exists():
        history = pd.read_csv(csv_path)
    else:
        raise FileNotFoundError(f"No cached history file for {row['run_name']} ({row['run_id']})")

    history = history.copy()
    history["run_name"] = row["run_name"]
    history["run_id"] = row["run_id"]
    if "_step" not in history.columns:
        if "charts/global_step" in history.columns:
            history["_step"] = history["charts/global_step"]
        elif "step" in history.columns:
            history["_step"] = history["step"]
        else:
            history["_step"] = np.arange(len(history), dtype=float)
    history["_step"] = pd.to_numeric(history["_step"], errors="coerce")
    history["step_millions"] = history["_step"] / 1_000_000
    return history


def load_cached_histories(selected_runs):
    frames = []
    total = len(selected_runs)
    for idx, row in enumerate(selected_runs.to_dict("records"), start=1):
        print(f"[{idx:>3}/{total}] loading {row['run_name']}", flush=True)
        try:
            frames.append(read_cached_history(row))
        except Exception as exc:
            print(f"    skipped: {type(exc).__name__}: {exc}")
    if not frames:
        raise RuntimeError("No histories could be loaded from the selected cache rows.")
    history = pd.concat(frames, ignore_index=True, sort=False)
    history = history.dropna(subset=["_step"])
    if PLOT_STEP_MAX_M is not None:
        history = history[history["step_millions"] <= float(PLOT_STEP_MAX_M)]
    return history.reset_index(drop=True)


def classify_run_name(run_name):
    name = str(run_name)
    seed_match = re.search(r"_s(\d+)(?:$|_)", name)
    number_match = re.match(r"^best_(\d+)_", name)
    variant = "include_neighbors" if "_neighbors_" in name or "include_neighbors" in name else "original"
    family = re.sub(r"_s\d+(?:$|_.*$)", "", name)
    family = family.replace("_neighbors_", "_")
    family_label = GINE_FAMILY_LABELS.get(family, family.replace("_", " "))
    return pd.Series({
        "variant": variant,
        "family": family,
        "family_label": family_label,
        "seed": int(seed_match.group(1)) if seed_match else np.nan,
        "run_number": number_match.group(1) if number_match else None,
    })


GINE_FAMILY_LABELS = {
    "best_000_shared_actor_gnn_gine_a4_no_concat_flat_critic_gnn_legacy_update": "000 no concat-flat",
    "best_00_shared_actor_gnn_gine_a4_concat_flat_critic_gnn_legacy_update": "00 baseline",
    "best_01_shared_actor_gnn_gine_a4_concat_flat_critic_gnn_optcritic": "01 optimized critic",
    "best_02_shared_actor_gnn_gine_a4_concat_flat_critic_mlp_legacy_update": "02 MLP critic",
    "best_03_nonshared_actor_gnn_gine_a4_concat_flat_critic_gnn_legacy_update": "03 non-shared actor",
    "best_04_shared_actor_gnn_light_gine_a4_concat_flat_critic_gnn_legacy_update": "04 light GINE",
    "best_05_shared_actor_gnn_gine_a4_entropy_decay_concat_flat_critic_gnn_legacy_update": "05 entropy decay",
    "best_06_shared_actor_gnn_gine_a4_no_node_id_concat_flat_critic_gnn_legacy_update": "06 no node ID",
    "best_07_shared_actor_gnn_gat_a4_concat_flat_critic_gnn_legacy_update": "07 GAT",
    "best_08_shared_actor_gnn_weighted_gcn_a4_concat_flat_critic_gnn_legacy_update": "08 weighted GCN",
}


In [ ]:
cache_index = load_cache_index()
selected_runs = select_cached_runs(cache_index)
run_meta = selected_runs[["run_name", "run_id"]].copy()
run_meta = pd.concat([run_meta, run_meta["run_name"].apply(classify_run_name)], axis=1)

print(f"Selected {len(selected_runs)} cached run(s) out of {len(cache_index)} available.")
display(run_meta.groupby(["variant", "family_label"], dropna=False).size().reset_index(name="runs"))

history_wide = load_cached_histories(selected_runs)
history_wide = history_wide.merge(run_meta, on=["run_name", "run_id"], how="left")
print(f"Loaded history shape: {history_wide.shape}")
history_wide.head()


## Metric Discovery And Reshaping

In [ ]:
ACTION0_PATTERN = re.compile(r"^train/frac_action_0_(agent_\d+)$")
ILLEGAL_PATTERN = re.compile(r"^train/illegal_action_rate_(agent_\d+)$")
ILLEGAL_COUNT_PATTERN = re.compile(r"^train/illegal_action_count_(agent_\d+)$")
ENTROPY_PATTERN = re.compile(r"^train/entropy_(agent_\d+)$")
NON_IDLE_COUNT_PATTERN = re.compile(r"^train/non_idle_agents_count_(\d+)_frac$")

ID_COLS = [
    "run_name",
    "run_id",
    "variant",
    "family",
    "family_label",
    "seed",
    "run_number",
    "_step",
    "step_millions",
]


def matching_columns(pattern, columns=None):
    columns = columns if columns is not None else history_wide.columns
    return sorted([col for col in columns if pattern.match(str(col))])


def melt_agent_metric(history, pattern, value_name):
    value_vars = matching_columns(pattern, history.columns)
    if not value_vars:
        return pd.DataFrame(columns=ID_COLS + ["metric", "agent", value_name])
    data = history[ID_COLS + value_vars].melt(
        id_vars=ID_COLS,
        value_vars=value_vars,
        var_name="metric",
        value_name=value_name,
    )
    data[value_name] = pd.to_numeric(data[value_name], errors="coerce")
    data = data.dropna(subset=[value_name])
    data["agent"] = data["metric"].str.extract(pattern.pattern)[0]
    return data


def melt_non_idle_count(history):
    value_vars = matching_columns(NON_IDLE_COUNT_PATTERN, history.columns)
    if not value_vars:
        return pd.DataFrame(columns=ID_COLS + ["metric", "non_idle_agents", "fraction"])
    data = history[ID_COLS + value_vars].melt(
        id_vars=ID_COLS,
        value_vars=value_vars,
        var_name="metric",
        value_name="fraction",
    )
    data["fraction"] = pd.to_numeric(data["fraction"], errors="coerce")
    data = data.dropna(subset=["fraction"])
    data["non_idle_agents"] = data["metric"].str.extract(NON_IDLE_COUNT_PATTERN.pattern)[0].astype(int)
    return data


def rolling_mean_by_group(data, value_col, group_cols, window=SMOOTH_WINDOW):
    if data.empty:
        return data.copy()
    out = data.sort_values(group_cols + ["_step"]).copy()
    if window is None or int(window) <= 1:
        out[f"{value_col}_smooth"] = out[value_col]
    else:
        out[f"{value_col}_smooth"] = out.groupby(group_cols, dropna=False)[value_col].transform(
            lambda values: values.rolling(int(window), min_periods=1).mean()
        )
    return out


def final_window_average(data, value_col, group_cols, window=FINAL_WINDOW_STEPS):
    if data.empty:
        return data.copy()
    sorted_data = data.sort_values(group_cols + ["_step"])
    tail = sorted_data.groupby(group_cols, dropna=False).tail(int(window))
    return tail.groupby(group_cols, dropna=False, as_index=False)[value_col].mean()


action0_long = melt_agent_metric(history_wide, ACTION0_PATTERN, "fraction_action0")
action0_long["fraction_non_idle"] = 1.0 - action0_long["fraction_action0"]
illegal_long = melt_agent_metric(history_wide, ILLEGAL_PATTERN, "illegal_rate")
illegal_count_long = melt_agent_metric(history_wide, ILLEGAL_COUNT_PATTERN, "illegal_count")
entropy_long = melt_agent_metric(history_wide, ENTROPY_PATTERN, "entropy")
non_idle_count_long = melt_non_idle_count(history_wide)

print("Metric rows:")
print(f"  action0_long:          {len(action0_long):,}")
print(f"  non_idle_count_long:   {len(non_idle_count_long):,}")
print(f"  illegal_long:          {len(illegal_long):,}")
print(f"  entropy_long:          {len(entropy_long):,}")
print("Agents:", sorted(action0_long["agent"].dropna().unique()))


## Aggregate Action Usage Over Training

In [ ]:
def aggregate_over_time(data, value_col, extra_group_cols):
    if data.empty:
        return data.copy()
    group_cols = ["variant", "family_label", *extra_group_cols, "_step", "step_millions"]
    return data.groupby(group_cols, dropna=False, as_index=False)[value_col].mean()


def plot_fraction_non_idle_over_time(families=None):
    data = action0_long.copy()
    if families is not None:
        data = data[data["family_label"].isin(families)]
    if data.empty:
        print("No train/frac_action_0_agent_* metrics found.")
        return None
    data = aggregate_over_time(data, "fraction_non_idle", ["agent"])
    data = rolling_mean_by_group(data, "fraction_non_idle", ["variant", "family_label", "agent"])
    fig = px.line(
        data,
        x="step_millions",
        y="fraction_non_idle_smooth",
        color="variant",
        line_dash="agent",
        facet_col="family_label",
        facet_col_wrap=2,
        labels={
            "step_millions": "steps (M)",
            "fraction_non_idle_smooth": "non-idle fraction",
            "variant": "variant",
            "agent": "agent",
        },
        title="Fraction of non-idle actions by agent",
    )
    fig.update_yaxes(range=[0, 1])
    fig.update_layout(template="plotly_white", height=max(520, 260 * max(1, data["family_label"].nunique() // 2 + 1)))
    save_figure(fig, "fraction_non_idle_by_agent_over_time")
    if SHOW_FIGURES:
        fig.show()
    return fig


def plot_action0_over_time(families=None):
    data = action0_long.copy()
    if families is not None:
        data = data[data["family_label"].isin(families)]
    if data.empty:
        print("No train/frac_action_0_agent_* metrics found.")
        return None
    data = aggregate_over_time(data, "fraction_action0", ["agent"])
    data = rolling_mean_by_group(data, "fraction_action0", ["variant", "family_label", "agent"])
    fig = px.line(
        data,
        x="step_millions",
        y="fraction_action0_smooth",
        color="variant",
        line_dash="agent",
        facet_col="family_label",
        facet_col_wrap=2,
        labels={
            "step_millions": "steps (M)",
            "fraction_action0_smooth": "action 0 fraction",
            "variant": "variant",
            "agent": "agent",
        },
        title="Fraction of action 0 by agent",
    )
    fig.update_yaxes(range=[0, 1])
    fig.update_layout(template="plotly_white", height=max(520, 260 * max(1, data["family_label"].nunique() // 2 + 1)))
    save_figure(fig, "fraction_action0_by_agent_over_time")
    if SHOW_FIGURES:
        fig.show()
    return fig


fig_non_idle_over_time = plot_fraction_non_idle_over_time()
fig_action0_over_time = plot_action0_over_time()


## Final-Window Action Summaries

In [ ]:
action_final = final_window_average(
    action0_long,
    "fraction_action0",
    ["run_name", "run_id", "variant", "family_label", "seed", "agent"],
)
action_final["fraction_non_idle"] = 1.0 - action_final["fraction_action0"]
action_final_grouped = action_final.groupby(
    ["variant", "family_label", "agent"],
    dropna=False,
    as_index=False,
).agg(
    fraction_action0=("fraction_action0", "mean"),
    fraction_non_idle=("fraction_non_idle", "mean"),
    runs=("run_id", "nunique"),
)

non_idle_final = final_window_average(
    non_idle_count_long,
    "fraction",
    ["run_name", "run_id", "variant", "family_label", "seed", "non_idle_agents"],
)
non_idle_final_grouped = non_idle_final.groupby(
    ["variant", "family_label", "non_idle_agents"],
    dropna=False,
    as_index=False,
).agg(fraction=("fraction", "mean"), runs=("run_id", "nunique"))

illegal_final = final_window_average(
    illegal_long,
    "illegal_rate",
    ["run_name", "run_id", "variant", "family_label", "seed", "agent"],
)
illegal_final_grouped = illegal_final.groupby(
    ["variant", "family_label", "agent"],
    dropna=False,
    as_index=False,
).agg(illegal_rate=("illegal_rate", "mean"), runs=("run_id", "nunique"))

print("Final-window non-idle fraction by family / variant / agent:")
display(action_final_grouped.sort_values(["family_label", "variant", "agent"]))


In [ ]:
def plot_final_non_idle_by_agent():
    if action_final_grouped.empty:
        print("No final-window action metrics to plot.")
        return None
    fig = px.bar(
        action_final_grouped,
        x="agent",
        y="fraction_non_idle",
        color="variant",
        barmode="group",
        facet_col="family_label",
        facet_col_wrap=2,
        hover_data=["runs", "fraction_action0"],
        labels={"fraction_non_idle": "non-idle fraction", "agent": "agent"},
        title=f"Final {FINAL_WINDOW_STEPS}-point mean: non-idle action fraction by agent",
    )
    fig.update_yaxes(range=[0, 1])
    fig.update_layout(template="plotly_white", height=max(520, 270 * max(1, action_final_grouped["family_label"].nunique() // 2 + 1)))
    save_figure(fig, "final_non_idle_fraction_by_agent")
    if SHOW_FIGURES:
        fig.show()
    return fig


def plot_final_joint_non_idle_distribution():
    if non_idle_final_grouped.empty:
        print("No train/non_idle_agents_count_*_frac metrics found.")
        return None
    fig = px.bar(
        non_idle_final_grouped,
        x="non_idle_agents",
        y="fraction",
        color="variant",
        barmode="group",
        facet_col="family_label",
        facet_col_wrap=2,
        hover_data=["runs"],
        labels={"non_idle_agents": "non-idle agents in same env step", "fraction": "fraction"},
        title=f"Final {FINAL_WINDOW_STEPS}-point mean: joint non-idle agent distribution",
    )
    fig.update_yaxes(range=[0, 1])
    fig.update_xaxes(dtick=1)
    fig.update_layout(template="plotly_white", height=max(520, 270 * max(1, non_idle_final_grouped["family_label"].nunique() // 2 + 1)))
    save_figure(fig, "final_joint_non_idle_distribution")
    if SHOW_FIGURES:
        fig.show()
    return fig


def plot_final_illegal_action_rate():
    if illegal_final_grouped.empty:
        print("No train/illegal_action_rate_agent_* metrics found.")
        return None
    fig = px.bar(
        illegal_final_grouped,
        x="agent",
        y="illegal_rate",
        color="variant",
        barmode="group",
        facet_col="family_label",
        facet_col_wrap=2,
        hover_data=["runs"],
        labels={"illegal_rate": "illegal action rate", "agent": "agent"},
        title=f"Final {FINAL_WINDOW_STEPS}-point mean: illegal action rate by agent",
    )
    fig.update_yaxes(rangemode="tozero")
    fig.update_layout(template="plotly_white", height=max(520, 270 * max(1, illegal_final_grouped["family_label"].nunique() // 2 + 1)))
    save_figure(fig, "final_illegal_action_rate_by_agent")
    if SHOW_FIGURES:
        fig.show()
    return fig


fig_final_non_idle = plot_final_non_idle_by_agent()
fig_joint_non_idle = plot_final_joint_non_idle_distribution()
fig_illegal_rate = plot_final_illegal_action_rate()


## Entropy And Exploration Context

In [ ]:
def plot_entropy_by_agent_over_time(families=None):
    data = entropy_long.copy()
    if families is not None:
        data = data[data["family_label"].isin(families)]
    if data.empty:
        print("No train/entropy_agent_* metrics found.")
        return None
    data = aggregate_over_time(data, "entropy", ["agent"])
    data = rolling_mean_by_group(data, "entropy", ["variant", "family_label", "agent"])
    fig = px.line(
        data,
        x="step_millions",
        y="entropy_smooth",
        color="variant",
        line_dash="agent",
        facet_col="family_label",
        facet_col_wrap=2,
        labels={"step_millions": "steps (M)", "entropy_smooth": "policy entropy"},
        title="Policy entropy by agent",
    )
    fig.update_yaxes(rangemode="tozero")
    fig.update_layout(template="plotly_white", height=max(520, 260 * max(1, data["family_label"].nunique() // 2 + 1)))
    save_figure(fig, "policy_entropy_by_agent_over_time")
    if SHOW_FIGURES:
        fig.show()
    return fig


fig_entropy = plot_entropy_by_agent_over_time()


## Optional Exact Action-ID Traces

The scalar history metrics tell us how often agents choose action 0 and how many agents act jointly, but they do not contain the full histogram over every discrete action id. For exact per-action-id histograms, runs must log rollout trace tables with `trace_rollout_actions = true` / `--trace-rollout-actions true`.

Set `FETCH_TRACE_TABLES_FROM_WANDB = True` above to try downloading those W&B tables for the selected runs.

In [ ]:
def _trace_table_ref_path(value):
    if isinstance(value, dict):
        return value.get("path") or value.get("artifact_path")
    if isinstance(value, str) and value.endswith(".table.json"):
        return value
    return None


def _read_wandb_table_json(path):
    payload = json.loads(Path(path).read_text(encoding="utf-8"))
    columns = payload.get("columns") or payload.get("schema", {}).get("columns")
    data = payload.get("data")
    if columns is None or data is None:
        raise ValueError(f"Not a recognized W&B table JSON: {path}")
    return pd.DataFrame(data, columns=columns)


def fetch_trace_tables_from_wandb(selected):
    if not FETCH_TRACE_TABLES_FROM_WANDB:
        print("Trace table fetch disabled. Set FETCH_TRACE_TABLES_FROM_WANDB = True to try W&B table download.")
        return pd.DataFrame()
    try:
        import wandb
    except ImportError as exc:
        raise ImportError("Install wandb first to fetch trace tables.") from exc

    api = wandb.Api(timeout=WANDB_API_TIMEOUT)
    frames = []
    for idx, row in enumerate(selected.to_dict("records"), start=1):
        run_name = row["run_name"]
        run_id = row["run_id"]
        print(f"[{idx:>3}/{len(selected)}] trace tables: {run_name}", flush=True)
        try:
            wb_run = api.run(f"{ENTITY}/{PROJECT}/{run_id}")
            for history_row in wb_run.scan_history(keys=["_step", *TRACE_TABLE_KEYS], page_size=2000):
                for key in TRACE_TABLE_KEYS:
                    table_path = _trace_table_ref_path(history_row.get(key))
                    if not table_path:
                        continue
                    target_dir = TRACE_CACHE_DIR / safe_name(run_id)
                    target_dir.mkdir(parents=True, exist_ok=True)
                    downloaded = wb_run.file(table_path).download(root=str(target_dir), replace=False)
                    table = _read_wandb_table_json(downloaded.name)
                    table["trace_key"] = key
                    table["trace_step"] = history_row.get("_step")
                    table["run_name"] = run_name
                    table["run_id"] = run_id
                    frames.append(table)
        except Exception as exc:
            print(f"    skipped: {type(exc).__name__}: {exc}")
    if not frames:
        return pd.DataFrame()
    traces = pd.concat(frames, ignore_index=True, sort=False)
    traces = traces.merge(run_meta, on=["run_name", "run_id"], how="left")
    return traces


def action_trace_to_long(trace_df):
    if trace_df.empty:
        return pd.DataFrame()
    action_cols = [col for col in trace_df.columns if re.match(r"^action_id_agent_\d+$", str(col))]
    if not action_cols:
        print("Trace tables were found, but no action_id_agent_* columns were present.")
        return pd.DataFrame()
    id_cols = [col for col in [
        "run_name", "run_id", "variant", "family_label", "seed", "trace_key", "trace_step",
        "global_step", "step", "env_idx", "episode", "episode_step", "non_idle_agents", "done",
    ] if col in trace_df.columns]
    long = trace_df[id_cols + action_cols].melt(
        id_vars=id_cols,
        value_vars=action_cols,
        var_name="agent_col",
        value_name="action_id",
    )
    long["agent"] = long["agent_col"].str.replace("action_id_", "", regex=False)
    long["action_id"] = pd.to_numeric(long["action_id"], errors="coerce")
    return long.dropna(subset=["action_id"])


def plot_action_id_histogram(trace_action_long, top_n=20):
    if trace_action_long.empty:
        print("No exact action-id traces available to plot.")
        return None
    counts = trace_action_long.groupby(
        ["variant", "family_label", "agent", "action_id"],
        dropna=False,
        as_index=False,
    ).size()
    counts["fraction"] = counts.groupby(["variant", "family_label", "agent"], dropna=False)["size"].transform(
        lambda values: values / max(values.sum(), 1)
    )
    top_actions = counts.sort_values("size", ascending=False).groupby(
        ["variant", "family_label", "agent"],
        dropna=False,
    ).head(int(top_n))
    fig = px.bar(
        top_actions,
        x="action_id",
        y="fraction",
        color="variant",
        barmode="group",
        facet_row="agent",
        facet_col="family_label",
        hover_data=["size"],
        labels={"action_id": "action id", "fraction": "fraction in trace"},
        title=f"Top {top_n} exact action ids from rollout traces",
    )
    fig.update_yaxes(rangemode="tozero")
    fig.update_layout(template="plotly_white", height=max(650, 260 * trace_action_long["agent"].nunique()))
    save_figure(fig, "trace_action_id_histogram")
    if SHOW_FIGURES:
        fig.show()
    return fig


trace_tables = fetch_trace_tables_from_wandb(selected_runs)
trace_action_long = action_trace_to_long(trace_tables)
fig_trace_actions = plot_action_id_histogram(trace_action_long)


## Quick Tables

In [ ]:
summary_tables = {
    "final_action_by_agent": action_final_grouped,
    "final_joint_non_idle": non_idle_final_grouped,
    "final_illegal_rate": illegal_final_grouped,
}

for name, table in summary_tables.items():
    if table.empty:
        continue
    path = FIG_DIR / f"{safe_name(name)}.csv"
    table.to_csv(path, index=False)
    print(f"Saved table: {path}")
